In [ ]:
# hide
import numpy as np
import pyquist as pq

In [ ]:
# hide
def feedback_delay(x: np.ndarray, M: int, g: float = 1.0) -> np.ndarray:
    """A feedback delay line with attenuation, y[n] = x[n] + g * y[n - M]."""
    buffer = np.zeros(M)              # the delay line, as a circular buffer
    i = 0
    y = np.zeros(len(x))
    for n in range(len(x)):
        y[n] = x[n] + g * buffer[i]   # attenuate y[n - M] on every trip
        buffer[i] = y[n]
        i = (i + 1) % M
    return y


def wet_ratio(db: float) -> float:
    """The wet ratio w that mixes the wet signal `db` decibels above the dry signal."""
    return 1 / (1 + pq.helper.db_to_amplitude(-db))

In [ ]:
yodel = pq.Audio.from_file("../assets/yodel.wav")
f_s = yodel.sample_rate
x = np.concatenate([yodel.samples[:, 0], np.zeros(5 * f_s)])   # room for the echoes
M = f_s                                       # a 1 second round trip
wet = feedback_delay(x, M, g=0.4)
w = wet_ratio(0.0)                            # equal parts wet and dry
pq.play(yodel, normalize=True)
pq.play(pq.Audio((1 - w) * x + w * wet, f_s), normalize=True)